In [7]:
import warnings
warnings.filterwarnings("ignore")
from dotenv import load_dotenv
import os
load_dotenv()
from elasticsearch import Elasticsearch

es = Elasticsearch(
    "https://localhost:9200",  
    basic_auth=("elastic", os.getenv("ELASTICSEARCH_PW")),  
    verify_certs=False  
)

print(es.info())


{'name': 'clyde', 'cluster_name': 'elasticsearch', 'cluster_uuid': 'HSc70bcgSbucwGX5aEqJ1g', 'version': {'number': '9.1.5', 'build_flavor': 'default', 'build_type': 'tar', 'build_hash': '90ee222e7e0136dd8ddbb34015538f3a00c129b7', 'build_date': '2025-10-02T22:07:12.966975992Z', 'build_snapshot': False, 'lucene_version': '10.2.2', 'minimum_wire_compatibility_version': '8.19.0', 'minimum_index_compatibility_version': '8.0.0'}, 'tagline': 'You Know, for Search'}


In [8]:
index_name = "articles"
es.options(ignore_status=[400]).indices.create(
    index=index_name,
    mappings={  
        "properties": {
            "content": {"type": "text"}
        }
    }
)

docs = [
    {"content": "Climate change affects water quality and agriculture."},
    {"content": "AI is transforming environmental monitoring systems."},
    {"content": "Water pollution and climate policy are global issues."},
]

for i, doc in enumerate(docs):
    es.index(index=index_name, id=i + 1, document=doc) 

es.indices.refresh(index=index_name)

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [9]:
query = {
    "query": {
        "match": {"content": "climate water"}
    },
    "explain": True
}

response = es.search(index=index_name, **query) 

for hit in response["hits"]["hits"]:
    print(f"Doc ID: {hit['_id']}")
    print(f"Score: {hit['_score']}")
    print(f"Content: {hit['_source']['content']} \n")

Doc ID: 1
Score: 0.9400072
Content: Climate change affects water quality and agriculture. 

Doc ID: 3
Score: 0.88810503
Content: Water pollution and climate policy are global issues. 

